# Lab 3.2: Content Moderation using Amazon Bedrock Data Automation

## In this notebook

We will explore:
- The content moderation features available in Amazon Bedrock Data Automation
- Practical methods for implementing visual content moderation
- Side-by-side comparison between Amazon Bedrock Data Automation and Amazon Rekognition for Content Moderation

We will complete the following steps:
- Create project in Amazon Bedrock Data Automation with Standard output and activate the Content Moderation feature specifically for image-based content
- Prepare S3 locations: `/input` with the images to moderate and `/output` to place job results
- Run [InvokeDataAutomationAsync](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_data-automation-runtime_InvokeDataAutomationAsync.html) API to initiate the image processing job for files located in the `/input` folder
- Review content moderation results in the `/output/*` folder

----
## [Amazon Bedrock Data Automation](https://docs.aws.amazon.com/bedrock/latest/userguide/bda.html)
**Amazon Bedrock Data Automation (BDA)** is a cloud-based service that simplifies the process of extracting valuable insights from unstructured content—such as documents, images, video, and audio. BDA leverages generative AI to automate the transformation of multi-modal data into structured formats, enabling developers to build applications and automate complex workflows with greater speed and accuracy.

One of the BDA use case is Content Moderation for image and video content. Content moderation detects inappropriate, unwanted, or offensive content in an image or vide. BDA supports 7 moderation categories: Explicit, Non-Explicit Nudity of Intimate parts and Kissing, Swimwear or Underwear, Violence, Drugs & Tobacco, Alcohol, Hate symbols. Explicit text in images/videos is not flagged.

## Content Moderation on AWS - Service comparison

| Features | Amazon Bedrock <br/> Data Automation | Amazon Rekognition <br/> Content Moderation |
| --- | --- | --- |
| Modality | Image | Image |
| Evaluated input | S3 bucket | Base64-encoded blob / S3 file |
| Input constraints | JPEG, PNG <br/> S3 file: up to 15 MB | JPEG, PNG <br/> Blob: up to 5 MB <br/> S3 file: up to 15 MB |
| Toxicity detection | [7 moderation categories](https://docs.aws.amazon.com/bedrock/latest/userguide/bda-ouput-image.html#content-moderation) | [3-level hierarchical categories](https://docs.aws.amazon.com/rekognition/latest/dg/moderation-api.html) |
| Toxicity dataset | Built-in | Built-in / Custom via adapters |
| Confidence score | Yes | Yes |
| API endpoint | [InvokeDataAutomationAsync](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_data-automation-runtime_InvokeDataAutomationAsync.html) <br/> [GetDataAutomationStatus](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_data-automation-runtime_InvokeDataAutomationAsync.html) | [DetectModerationLabels](https://docs.aws.amazon.com/rekognition/latest/APIReference/API_DetectModerationLabels.html) |
| Cost | Per image | Per image |
| Regions | [Supported regions](https://docs.aws.amazon.com/bedrock/latest/userguide/bda-cris.html) | [Supported regions](https://docs.aws.amazon.com/general/latest/gr/rekognition.html#rekognition_region) |
| Additional features | Image summary, Logo detection, <br/> IAB taxonomy, Image text detection | Face & identity detection, <br/> Label & text detection |

----
## Demo Data Flow
<img src="diagrams/ContentModerationWithBDA.png" alt="Content Moderation With Amazon Bedrock Guardrail">

----
## Pre-requisite steps

Load libraries and update Amazon SageMaker Notebook IAM role with the nessesary permissions

In [ ]:
import boto3
import json

In [ ]:
iam = boto3.client('iam')
sts = boto3.client('sts')
s3 = boto3.client('s3')
bda = boto3.client('bedrock-data-automation')
bda_runtime = boto3.client('bedrock-data-automation-runtime')
session = boto3.session.Session()

region = session.region_name
account_id = boto3.client('sts').get_caller_identity().get('Account')

In [ ]:
# Add required permissions to current SageMaker Notebook role
from sagemaker import get_execution_role

policy_name = "CustomBDAPolicy"
policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:CreateDataAutomationProject",
                "bedrock:GetDataAutomationProject",
                "bedrock:InvokeDataAutomationAsync",
                "bedrock:DeleteDataAutomationProject"
            ],
            "Resource": [
                f"arn:aws:bedrock:{region}:{account_id}:data-automation-project/*",
                f"arn:aws:bedrock:*:{account_id}:data-automation-profile/*"
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:ListDataAutomationProjects"
            ],
            "Resource": [
                "*"
            ]
        },
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:GetDataAutomationStatus"
            ],
            "Resource": [
                f"arn:aws:bedrock:{region}:{account_id}:data-automation-invocation/*"
            ]
        }
    ]
}

# Get current execution role
current_role = get_execution_role()
print(f"Current execution role: {current_role}")

role_name = current_role.split('/')[-1]
response = iam.put_role_policy(
    RoleName=role_name,
    PolicyName=policy_name,
    PolicyDocument=json.dumps(policy_document)
)
print(f"Successfully added inline policy {policy_name} to role {role_name}")

In [ ]:
bucket_name = account_id + "-" + region + "-" + "bda"

if region == "us-east-1":
    bucket_responese = s3.create_bucket(Bucket=bucket_name)
else:
    bucket_responese = s3.create_bucket(
        Bucket=bucket_name,
        CreateBucketConfiguration={'LocationConstraint': region}
        )
file_name = "man-smoking-cigarette.jpg"
file_location = f"images/{file_name}"
s3.upload_file(file_location, bucket_name, f"input/{file_name}")

----
## Create Bedrock Data Automation Project

In [ ]:
%run bda_utils.py 

In [ ]:
bda_project_arn = create_bda_project(
    project_name="marketing-project",
    project_description="Project for processing marketing images",
)
print(bda_project_arn)

## Run content moderation through Bedrock Data Automation

Now that your project is configured, you can begin analyzing images using the [InvokeDataAutomationAsync](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_data-automation-runtime_InvokeDataAutomationAsync.html) operation.

This API call initiates the asynchronous processing of the sample file in a specified S3 bucket. The API accepts the project ARN and the file to be processed, then starts the asynchronous processing job. A job ID is returned for tracking the process.

In [ ]:
input_s3_uri = f"s3://{bucket_name}/input/{file_name}" # File
output_s3_uri = f"s3://{bucket_name}/output" # Folder
bda_profile_arn = f"arn:aws:bedrock:{region}:{account_id}:data-automation-profile/us.data-automation-v1"

params = {
    'inputConfiguration': {
        's3Uri': input_s3_uri
    },
    'outputConfiguration': {
        's3Uri': output_s3_uri
    },
    'dataAutomationConfiguration': {
        'dataAutomationProjectArn': bda_project_arn,
        'stage': 'LIVE' #'DEVELOPMENT'
    },
    'dataAutomationProfileArn': bda_profile_arn
}

response = bda_runtime.invoke_data_automation_async(**params)
invocation_arn = response['invocationArn']
print(invocation_arn)

The [GetDataAutomationStatus](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_data-automation-runtime_GetDataAutomationStatus.html) API allows you to monitor the progress of your job and access the results once processing is complete. The API accepts the invocation ARN returned by InvokeDataAutomationAsync. It checks the current status of the job and returns relevant information. Once the job is complete, it provides the location of the results in S3.

Now lets retrieve BDA job results in order to retrieve S3 location with image moderation results.
- If the job is still in progress, it returns the current state (e.g., "InProgress"). 
- If the job is complete, it returns "Success" along with the S3 location of the results. 
- If there was an error, it returns "ServiceError" or "ClientError" with error details.

In [ ]:
# Retrieve BDA job results

import time
print("Wait until successful response with S3 uri location of the job results")
while True:
    response = bda_runtime.get_data_automation_status(
        invocationArn=invocation_arn
    )
    status = response['status']
    if status not in ['Created', 'InProgress']:
        print(f" {status}")
        break 
    else:
        print(".", end='', flush=True)
        time.sleep(15)
print(json.dumps(response, indent=2))

Lastly, retrieve BDA image processing results with `summary` and `content_moderation` nodes

In [ ]:
# Retrieve BDA image processing results which include summary and content_moderation node

job_metadata_s3_uri = response['outputConfiguration']['s3Uri']
job_metadata = get_json_object_from_s3_uri(job_metadata_s3_uri)

for segment in job_metadata['output_metadata']:
    job_s3_uri = segment['segment_metadata'][0]['standard_output_path']
    job_output = get_json_object_from_s3_uri(job_s3_uri)
    print(json.dumps(job_output['image'], indent=2))

# Summary

In this notebook, we demonstrated how to leverage [Amazon Bedrock Data Automation (BDA)](https://docs.aws.amazon.com/bedrock/latest/userguide/bda.html) to identify unwanted content across supported [7 moderation categories](https://docs.aws.amazon.com/bedrock/latest/userguide/bda-ouput-image.html#content-moderation) by enabling its Content Moderation feature for images. We learned how to structure a BDA project and invoke API [InvokeDataAutomationAsync](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_data-automation-runtime_InvokeDataAutomationAsync.html) asynchronously to process input image files and review the results within `content_moderation` node in the output path.

# Cleanup

To clean up the Amazon BDA resources created in this notebook, you can execute the section **Clean up Amazon Bedrock Data Automation** in the [`04_Clean_Up.ipynb`](04_Clean_Up.ipynb) notebook. If you're running these notebooks as part of an AWS-led workshop where temporary AWS accounts are provided for you, this cleanup will be done automatically for you. Otherwise, if you're running this notebook in a personal or work account, be sure to run the [`04_Clean_Up.ipynb`](04_Clean_Up.ipynb) notebook to shutdown resources that can create ongoing AWS charges.

# Next Steps

In these notebooks we have focused on optimizing the "who" and "what" in the email marketing campaigns for our ficticious travel company. Optimizing the "who" involved using Amazon Personalize's user segmentation recipe to train a ML model that identified users with an affinity for the trip we wanted to promote. Then we turned to optimizing the "what" by using Amazon Bedrock to generate an appealing email subject, body, and banner image. What we didn't cover is the "how" to deliver these messages. AWS provides services such as [Amazon Pinpoint](https://aws.amazon.com/pinpoint/) and [Amazon Simple Email Service (SES)](https://aws.amazon.com/ses/) that make it easy to deliver, manage, and measure the email delivery process.